In [ ]:
!pip install pdfplumber

In [ ]:
import pdfplumber
import pandas as pd
import re

pdf_path = 'surveillance-report121.pdf'

with pdfplumber.open(pdf_path) as pdf:
    page1_text = pdf.pages[0].extract_text()
    page2_text = pdf.pages[1].extract_text()

full_text = page1_text + '\n' + page2_text
lines = full_text.split('\n')

data_rows = []

for line in lines:
    # Skipping header/footer lines
    if any(skip in line for skip in ['Table 2', 'Beer Wine Spirits', 'State or other',
                                      'geographic area', 'capita ethanol', 'Volume Ethanol',
                                      'capita capita', 'Decile values', 'Numbers may not sum',
                                      'Regions', 'apply only']):
        continue

    # Check if line looks like a state (starts with capital letter and has numbers)
    if re.match(r'^[A-Z]', line) and re.search(r'\d', line):
        # Only remove 2+ dots in a row, remove isolated dots, remove single dots with spaces, but keep decimal points
        cleaned = re.sub(r'\.{2,}', ' ', line)
        cleaned = re.sub(r'\s\.+\s', ' ', cleaned)
        cleaned = re.sub(r'\s\.\s', ' ', cleaned)

        parts = cleaned.split()

        state_parts = []
        numbers = []
        found_first_number = False

        for part in parts:
            # Check if it's a number
            if re.match(r'^\d+[,\d]*\.?\d*$', part) or re.match(r'^\d*\.\d+$', part):
                numbers.append(part.replace(',', ''))
                found_first_number = True
            else:
                # Only add to state name if we haven't found any numbers yet
                if not found_first_number:
                    state_parts.append(part)

        # 12 numbers per state
        if len(numbers) == 12:
            state_name = ' '.join(state_parts)
            data_rows.append([state_name] + numbers)

columns = ['State',
           'Beer_Volume', 'Beer_Ethanol', 'Beer_PerCapita',
           'Wine_Volume', 'Wine_Ethanol', 'Wine_PerCapita',
           'Spirits_Volume', 'Spirits_Ethanol', 'Spirits_PerCapita',
           'Total_Ethanol', 'Total_PerCapita', 'Decile']

df = pd.DataFrame(data_rows, columns=columns)

for col in columns[1:]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.to_csv('alcohol_consumption_2022.csv', index=False)

print(f"CSV created successfully!")
print(f"\nShape: {df.shape}")
print(f"\nAll {len(df)} states:")
for state in df['State'].tolist():
    print(f"  - {state}")

CSV created successfully!

Shape: (51, 13)

All 51 states:
  - Alabama
  - Alaska
  - Arizona
  - Arkansas
  - California
  - Colorado
  - Connecticut
  - Delaware
  - Dist. of Columb.
  - Florida
  - Georgia
  - Hawaii
  - Idaho
  - Illinois
  - Indiana
  - Iowa
  - Kansas
  - Kentucky
  - Louisiana
  - Maine
  - Maryland
  - Massachusetts
  - Michigan
  - Minnesota
  - Mississippi
  - Missouri
  - Montana
  - Nebraska
  - Nevada
  - New Hampshire
  - New Jersey
  - New Mexico
  - New York
  - North Carolina
  - North Dakota
  - Ohio
  - Oklahoma
  - Oregon
  - Pennsylvania
  - Rhode Island
  - South Carolina
  - South Dakota
  - Tennessee
  - Texas
  - Utah
  - Vermont
  - Virginia
  - Washington
  - West Virginia
  - Wisconsin
  - Wyoming


In [ ]:
# Checking raw file type
with open('2022_TRAFFIC_ALCOHOLDEATHS.xls', 'rb') as f:
    header = f.read(8)
    print("File header (hex):", header.hex())
    print("File header (ascii):", header)

with open('2022_TRAFFIC_ALCOHOLDEATHS.xls', 'r', encoding='latin-1', errors='ignore') as f:
    print("\nFirst 500 characters:")
    print(f.read(500))

File header (hex): 0948696768657374
File header (ascii): b'\tHighest'

First 500 characters:
	Highest Blood Alcohol Concentration in Crash											
	BAC = 0.00		BAC = .01--.07		BAC = .08+		BAC=.01+		Total Killed			
State	Number	Percent	Number	Percent	Number	Percent	Number	Percent	Number	Percent		

Alabama	666	67	45	5	276	28	321	32	988	100	
Alaska	61	75	1	1	20	24	21	25	82	100	
Arizona	788	60	72	5	455	34	527	40	1320	100	
Arkansas	440	69	48	7	150	23	197	31	637	100	
California	2849	63	263	6	1419	31	1683	37	4539	100	
Colorado	449	59	54	7	259	34	313	41	764	100	
Connecticut	212	58	25	7	129	3


In [ ]:
# Read as tab-delimited CSV
df = pd.read_csv('2022_TRAFFIC_ALCOHOLDEATHS.xls', sep='\t')

df.to_csv('blood_alcohol_data_and_deaths.csv', index=False)

print("CSV created successfully!")
print(f"\nShape: {df.shape}")
print(f"\nFirst 10 rows:")
print(df.head(10))

CSV created successfully!

Shape: (54, 13)

First 10 rows:
    Unnamed: 0 Highest Blood Alcohol Concentration in Crash Unnamed: 2  \
0          NaN                                   BAC = 0.00        NaN   
1        State                                       Number    Percent   
2      Alabama                                          666         67   
3       Alaska                                           61         75   
4      Arizona                                          788         60   
5     Arkansas                                          440         69   
6   California                                         2849         63   
7     Colorado                                          449         59   
8  Connecticut                                          212         58   
9     Delaware                                          101         63   

       Unnamed: 3 Unnamed: 4  Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8  \
0  BAC = .01--.07        NaN  BAC = .08+        N

In [ ]:
# Checking structure of the two csv's
alcohol = pd.read_csv('alcohol_consumption_2022.csv')
deaths = pd.read_csv('blood_alcohol_data_and_deaths.csv')

print("Alcohol columns:")
print(alcohol.columns.tolist())

print("\n" + "="*80)
print("\nDeaths columns:")
print(deaths.columns.tolist())

print("\n" + "="*80)
print("\nFirst few rows of deaths data:")
print(deaths.head(10))

Alcohol columns:
['State', 'Beer_Volume', 'Beer_Ethanol', 'Beer_PerCapita', 'Wine_Volume', 'Wine_Ethanol', 'Wine_PerCapita', 'Spirits_Volume', 'Spirits_Ethanol', 'Spirits_PerCapita', 'Total_Ethanol', 'Total_PerCapita', 'Decile']


Deaths columns:
['Unnamed: 0', 'Highest Blood Alcohol Concentration in Crash', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12']


First few rows of deaths data:
    Unnamed: 0 Highest Blood Alcohol Concentration in Crash Unnamed: 2  \
0          NaN                                   BAC = 0.00        NaN   
1        State                                       Number    Percent   
2      Alabama                                          666         67   
3       Alaska                                           61         75   
4      Arizona                                          788         60   
5     Arkansas                                          

In [ ]:
# Load deaths data, skip first row, use row 1 as header
deaths = pd.read_csv('blood_alcohol_data_and_deaths.csv', skiprows=1)

# The first column should be State
deaths.columns = ['State', 'BAC_0.00_Number', 'BAC_0.00_Percent',
                  'BAC_0.01-0.07_Number', 'BAC_0.01-0.07_Percent',
                  'BAC_0.08+_Number', 'BAC_0.08+_Percent',
                  'BAC_0.01+_Number', 'BAC_0.01+_Percent',
                  'Total_Number', 'Total_Percent', 'Unnamed_11', 'Unnamed_12']

# Remove the extra columns and any empty rows
deaths = deaths[['State', 'BAC_0.00_Number', 'BAC_0.00_Percent',
                 'BAC_0.01-0.07_Number', 'BAC_0.01-0.07_Percent',
                 'BAC_0.08+_Number', 'BAC_0.08+_Percent',
                 'BAC_0.01+_Number', 'BAC_0.01+_Percent',
                 'Total_Number', 'Total_Percent']].copy()

# Remove NaN states
deaths = deaths[deaths['State'].notna()]

print("First 10 rows:")
print(deaths.head(10))

print("\nStates in deaths data:")
print(sorted(deaths['State'].unique()))

# Load alcohol data
alcohol = pd.read_csv('alcohol_consumption_2022.csv')

# Mapping for District of Columbia
state_mapping = {
    'Dist. of Columb.': 'District of Columbia'
}

# Mapping to alcohol data w/ abbreviated version
alcohol['State'] = alcohol['State'].replace(state_mapping)

# Merge
merged = pd.merge(alcohol, deaths, on='State', how='inner')

print(f"\nMerged shape: {merged.shape}")
print(f"States successfully merged: {len(merged)}")

merged.to_csv('merged_alcohol_deaths.csv', index=False)
print("\nMerged data saved!")

First 10 rows:
                  State BAC_0.00_Number BAC_0.00_Percent BAC_0.01-0.07_Number  \
0                 State          Number          Percent               Number   
1               Alabama             666               67                   45   
2                Alaska              61               75                    1   
3               Arizona             788               60                   72   
4              Arkansas             440               69                   48   
5            California            2849               63                  263   
6              Colorado             449               59                   54   
7           Connecticut             212               58                   25   
8              Delaware             101               63                   10   
9  District of Columbia              20               63                    2   

  BAC_0.01-0.07_Percent BAC_0.08+_Number BAC_0.08+_Percent BAC_0.01+_Number  \
0             

In [ ]:
# Population data
pop = pd.read_csv('NST-EST2022-ALLDATA.csv')

print("Population data columns:")
print(pop.columns.tolist())
print("\nFirst few rows:")
print(pop.head())

print("\nLooking for state column...")
for col in pop.columns:
    if 'name' in col.lower() or 'state' in col.lower():
        print(f"Found: {col}")
        print(pop[col].head(10))

Population data columns:
['SUMLEV', 'REGION', 'DIVISION', 'STATE', 'NAME', 'ESTIMATESBASE2020', 'POPESTIMATE2020', 'POPESTIMATE2021', 'POPESTIMATE2022', 'NPOPCHG_2020', 'NPOPCHG_2021', 'NPOPCHG_2022', 'BIRTHS2020', 'BIRTHS2021', 'BIRTHS2022', 'DEATHS2020', 'DEATHS2021', 'DEATHS2022', 'NATURALCHG2020', 'NATURALCHG2021', 'NATURALCHG2022', 'INTERNATIONALMIG2020', 'INTERNATIONALMIG2021', 'INTERNATIONALMIG2022', 'DOMESTICMIG2020', 'DOMESTICMIG2021', 'DOMESTICMIG2022', 'NETMIG2020', 'NETMIG2021', 'NETMIG2022', 'RESIDUAL2020', 'RESIDUAL2021', 'RESIDUAL2022', 'RBIRTH2021', 'RBIRTH2022', 'RDEATH2021', 'RDEATH2022', 'RNATURALCHG2021', 'RNATURALCHG2022', 'RINTERNATIONALMIG2021', 'RINTERNATIONALMIG2022', 'RDOMESTICMIG2021', 'RDOMESTICMIG2022', 'RNETMIG2021', 'RNETMIG2022']

First few rows:
   SUMLEV REGION DIVISION  STATE              NAME  ESTIMATESBASE2020  \
0      10      0        0      0     United States          331449520   
1      20      1        0      0  Northeast Region           5760

In [ ]:
# Load all datasets
pop = pd.read_csv('NST-EST2022-ALLDATA.csv')
merged = pd.read_csv('merged_alcohol_deaths.csv')

# Filter for individual states (SUMLEV == 40 is state level)
pop_states = pop[pop['SUMLEV'] == 40][['NAME', 'POPESTIMATE2022']].copy()
pop_states.columns = ['State', 'Population_2022']

print("States in population data:")
print(sorted(pop_states['State'].unique()))

# Fix state name mismatches
state_mapping = {
    'Dist. of Columb.': 'District of Columbia'
}

# Check what states we have in merged data
print("\nStates in merged data:")
print(sorted(merged['State'].unique()))

# Merge population data with existing merged data
final = pd.merge(merged, pop_states, on='State', how='left')

# Convert columns to numeric
final['Total_Number'] = pd.to_numeric(final['Total_Number'], errors='coerce')
final['BAC_0.00_Number'] = pd.to_numeric(final['BAC_0.00_Number'], errors='coerce')
final['BAC_0.01-0.07_Number'] = pd.to_numeric(final['BAC_0.01-0.07_Number'], errors='coerce')
final['BAC_0.08+_Number'] = pd.to_numeric(final['BAC_0.08+_Number'], errors='coerce')
final['BAC_0.01+_Number'] = pd.to_numeric(final['BAC_0.01+_Number'], errors='coerce')

# Calculate deaths per capita (per 100K ppl)
final['Total_Deaths_PerCapita'] = (final['Total_Number'] / final['Population_2022']) * 100000
final['BAC_0.00_Deaths_PerCapita'] = (final['BAC_0.00_Number'] / final['Population_2022']) * 100000
final['BAC_0.01-0.07_Deaths_PerCapita'] = (final['BAC_0.01-0.07_Number'] / final['Population_2022']) * 100000
final['BAC_0.08+_Deaths_PerCapita'] = (final['BAC_0.08+_Number'] / final['Population_2022']) * 100000
final['BAC_0.01+_Deaths_PerCapita'] = (final['BAC_0.01+_Number'] / final['Population_2022']) * 100000

final.to_csv('final_alcohol_analysis.csv', index=False)

print(f"\nFinal dataset shape: {final.shape}")
print(f"\nStates with data: {final['State'].nunique()}")
print(f"\nStates missing population data:")
print(final[final['Population_2022'].isna()]['State'].tolist())

print("\nTop 10 states by total deaths per capita:")
print(final.nlargest(10, 'Total_Deaths_PerCapita')[['State', 'Total_Number', 'Population_2022', 'Total_Deaths_PerCapita']])

print("\nDataset saved as 'final_alcohol_analysis.csv'!")

States in population data:
['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Puerto Rico', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming']

States in merged data:
['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minneso